In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../..')
from common.style import PRIMARY, PRIMARY_DARK, ACCENT, ACCENT_LIGHT, AXIS_COLOR, GRID_COLOR

da = pd.read_excel('../../utaut2_dataset/utaut2_subsample.xlsx')

construct_names = {
    'PU': 'Воспринимаемая полезность',
    'PEOU': 'Лёгкость использования',
    'SI': 'Социальное влияние',
    'FC': 'Поддерживающие условия',
    'HM': 'Гедоническая мотивация',
    'PV': 'Ценность цены',
    'HAB': 'Привычка',
    'BI': 'Поведенческое намерение'
}

constructs = ['PU', 'PEOU', 'SI', 'FC', 'HM', 'PV', 'HAB', 'BI']
predictors = ['PU', 'PEOU', 'SI', 'FC', 'HM', 'PV', 'HAB']

# Пункты конструктов (нужны для Cronbach alpha)
df_full = pd.read_excel('../../utaut2_dataset/new_result.xlsx')
cols = df_full.columns.tolist()
construct_items = {
    'PU':   cols[13:16],
    'PEOU': cols[16:19],
    'SI':   cols[19:22],
    'FC':   cols[22:25],
    'HM':   cols[25:28],
    'PV':   cols[28:31],
    'HAB':  cols[31:34],
    'BI':   cols[34:37],
}

print(f'Аналитическая подвыборка UTAUT2: {len(da)}')

Аналитическая подвыборка UTAUT2: 164


In [2]:
norm_rows = []
for c in constructs:
    w_stat, p_value = stats.shapiro(da[c])
    norm_rows.append({
        'Код': c,
        'Конструкт': construct_names[c],
        'Shapiro-Wilk W': round(w_stat, 4),
        'p-value': f'{p_value:.6f}',
        'Асимметрия': round(da[c].skew(), 3),
        'Эксцесс': round(da[c].kurtosis(), 3)
    })

norm_df = pd.DataFrame(norm_rows)
norm_df


,Код,Конструкт,Shapiro-Wilk W,p-value,Асимметрия,Эксцесс
0,PU,Воспринимаемая полезность,0.9686,0.000895,-0.300,0.016
1,PEOU,Лёгкость использования,0.9420,0.000003,-0.420,-0.136
2,SI,Социальное влияние,0.9558,0.000047,-0.451,1.605
3,FC,Поддерживающие условия,0.9085,0.000000,-0.563,1.019
4,HM,Гедоническая мотивация,0.9437,0.000004,-0.579,0.966
5,PV,Ценность цены,0.9574,0.000066,0.437,0.054
6,HAB,Привычка,0.9346,0.000001,0.136,-1.267
7,BI,Поведенческое намерение,0.9690,0.000985,-0.158,-0.394


In [3]:
print('Проверка распределений конструктов:\n')

for _, row in norm_df.iterrows():
    print(
        f"{row['Код']}: W = {row['Shapiro-Wilk W']:.4f}, "
        f"p = {row['p-value']}, "
        f"асимметрия = {row['Асимметрия']:.3f}, "
        f"эксцесс = {row['Эксцесс']:.3f}"
    )

print('\nВывод:')
print('Shapiro-Wilk отвергает строгую нормальность распределения для конструктов UTAUT2.')
print('При этом значения асимметрии и эксцесса остаются в умеренных пределах.')
print('С учётом объёма аналитической подвыборки это позволяет использовать OLS-регрессию как приближение, сохраняя осторожность при интерпретации результатов.')


Проверка распределений конструктов:

PU: W = 0.9686, p = 0.000895, асимметрия = -0.300, эксцесс = 0.016
PEOU: W = 0.9420, p = 0.000003, асимметрия = -0.420, эксцесс = -0.136
SI: W = 0.9558, p = 0.000047, асимметрия = -0.451, эксцесс = 1.605
FC: W = 0.9085, p = 0.000000, асимметрия = -0.563, эксцесс = 1.019
HM: W = 0.9437, p = 0.000004, асимметрия = -0.579, эксцесс = 0.966
PV: W = 0.9574, p = 0.000066, асимметрия = 0.437, эксцесс = 0.054
HAB: W = 0.9346, p = 0.000001, асимметрия = 0.136, эксцесс = -1.267
BI: W = 0.9690, p = 0.000985, асимметрия = -0.158, эксцесс = -0.394

Вывод:
Shapiro-Wilk отвергает строгую нормальность распределения для конструктов UTAUT2.
При этом значения асимметрии и эксцесса остаются в умеренных пределах.
С учётом объёма аналитической подвыборки это позволяет использовать OLS-регрессию как приближение, сохраняя осторожность при интерпретации результатов.
